# Speech Analysis

## Installing and importing the libraries.

In [11]:
!apt-get -qq install -y ffmpeg
%pip install numpy scipy librosa soundfile matplotlib pystoi pandas pydub -q

In [ ]:
import csv
import math
import re
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from IPython.display import Audio, Markdown, display
from pydub import AudioSegment
from pystoi import stoi
from scipy.signal import butter, sosfiltfilt, sosfreqz, stft

# Google Colab paths. Keep all generated artifacts under /content/outputs.
data_dir = Path("/content/data")
noise_dir = Path("/content/Noise")
output_dir = Path("/content/outputs")
clean_audio_dir = output_dir / "clean_wav"
for folder in (data_dir, noise_dir, output_dir, clean_audio_dir):
    folder.mkdir(parents=True, exist_ok=True)

TARGET_SAMPLE_RATE = 16000
STFT_N_FFT = 1024
STFT_HOP_LENGTH = 256
NOISE_SOURCE_STEMS = {"white", "machinegun", "babble"}
print("Libraries imported successfully.")

Libraries imported successfully!


In [ ]:
audio_extensions = {".wav", ".mp3", ".m4a", ".aac", ".ogg", ".flac", ".wma", ".aiff", ".aif"}
source_files = sorted(
    path for path in data_dir.iterdir()
    if path.is_file() and path.suffix.lower() in audio_extensions
)
if not source_files:
    raise FileNotFoundError(f"No supported audio files found in {data_dir}")

# Normalize every clean input to mono, 16 kHz, 16-bit PCM WAV.
for audio_path in source_files:
    wav_path = clean_audio_dir / f"{audio_path.stem}.wav"
    audio = (
        AudioSegment.from_file(audio_path)
        .set_frame_rate(TARGET_SAMPLE_RATE)
        .set_channels(1)
        .set_sample_width(2)
    )
    audio.export(wav_path, format="wav", parameters=["-acodec", "pcm_s16le"])
    print(f"Normalized {audio_path.name} -> {wav_path}")

assert list(clean_audio_dir.glob("*.wav")), "No normalized clean WAV files were created."

### Extracting Raw Information from the audio and plotting graph

In [ ]:
def analyze_audio_directory(input_dir, output_dir="/content/outputs/Audio_waveform_graph"):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    if not input_dir.is_dir():
        raise NotADirectoryError(f"Audio folder not found: {input_dir}")

    output_dir.mkdir(parents=True, exist_ok=True)
    audio_extensions = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".wma"}

    audio_files = sorted(
        path for path in input_dir.iterdir()
        if path.is_file() and path.suffix.lower() in audio_extensions
    )

    if not audio_files:
        print(f"No supported audio files found in {input_dir}")
        return []

    results = []

    for audio_path in audio_files:
        audio, sample_rate = librosa.load(audio_path, sr=None, mono=False)

        if audio.ndim == 1:
            audio = audio[np.newaxis, :]

        channel_count, sample_count = audio.shape
        duration = sample_count / sample_rate

        metadata = {
            "file": audio_path.name,
            "format": audio_path.suffix.lower().lstrip("."),
            "sample_rate_hz": sample_rate,
            "channels": channel_count,
            "samples_per_channel": sample_count,
            "duration_seconds": duration,
            "peak_amplitude": float(np.max(np.abs(audio))),
            "rms_amplitude": float(np.sqrt(np.mean(np.square(audio, dtype=np.float64)))),
        }

        time = np.arange(sample_count) / sample_rate
        fig, axes = plt.subplots(
            channel_count, 1,
            figsize=(14, max(3, 2.5 * channel_count)),
            sharex=True,
            squeeze=False,
        )

        for channel in range(channel_count):
            axes[channel, 0].plot(time, audio[channel], linewidth=0.5)
            axes[channel, 0].set_ylabel(f"Channel {channel + 1}")
            axes[channel, 0].grid(True, alpha=0.3)

        axes[-1, 0].set_xlabel("Time (seconds)")
        fig.suptitle(f"Waveform: {audio_path.name}")
        fig.tight_layout()

        graph_path = output_dir / f"{audio_path.stem}_waveform.png"
        fig.savefig(graph_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        results.append({"metadata": metadata, "graph_path": graph_path})
        print(f"{audio_path.name}: {duration:.2f} seconds — graph saved to {graph_path}")

    return results

### Inspecting the data and calling the Plotting graph

In [ ]:
results = analyze_audio_directory(clean_audio_dir)

### Step 1 analysis

Two voice recordings were made. We changed both files to mono WAV files with a sample rate of 16,000 Hz.

| Recording | Duration | Sample rate |
|---|---:|---:|
| Speech 1 | 10.496 s | 16,000 Hz |
| Speech 2 | 10.112 s | 16,000 Hz |

These original files were used to compare the changed, noisy, and filtered audio.

### Calculating STFT and generating sig1

In [ ]:
def get_audio_stft_features(audio_dir, n_fft=STFT_N_FFT, hop_length=STFT_HOP_LENGTH):
    audio_dir = Path(audio_dir)

    if not audio_dir.is_dir():
        raise NotADirectoryError(f"Audio directory not found: {audio_dir}")

    extensions = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".wma"}
    audio_files = sorted(
        path for path in audio_dir.iterdir()
        if path.is_file() and path.suffix.lower() in extensions
    )

    results = {}

    for path in audio_files:
        audio, sample_rate = librosa.load(path, sr=None, mono=False)

        if audio.ndim == 1:  # Mono: (frequency bins, time frames)
            stft = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
        else:  # Multi-channel: (channels, frequency bins, time frames)
            stft = np.stack([
                librosa.stft(channel, n_fft=n_fft, hop_length=hop_length)
                for channel in audio
            ])

        results[path.name] = {
            "stft": stft,
            "magnitude": np.abs(stft),
            "phase": np.angle(stft),  # Radians
            "sample_rate": sample_rate,
            "n_fft": n_fft,
            "hop_length": hop_length,
        }

    if not results:
        print(f"No supported audio files found in {audio_dir}")

    return results

In [ ]:
def plot_magnitude_phase_spectrograms(
    audio_dir,
    output_dir="/content/outputs/MG_PH_Spectrogram",
    n_fft=STFT_N_FFT,
    hop_length=STFT_HOP_LENGTH,
):
    audio_dir = Path(audio_dir)
    output_dir = Path(output_dir)

    if not audio_dir.is_dir():
        raise NotADirectoryError(f"Audio directory not found: {audio_dir}")

    extensions = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".wma"}
    audio_files = sorted(
        path for path in audio_dir.iterdir()
        if path.is_file() and path.suffix.lower() in extensions
    )

    if not audio_files:
        raise FileNotFoundError(f"No supported audio files found in {audio_dir}")

    output_dir.mkdir(parents=True, exist_ok=True)
    saved_graphs = []

    for audio_path in audio_files:
        # Mono simplifies the plot when an input file has multiple channels.
        audio, sample_rate = librosa.load(audio_path, sr=None, mono=True)

        stft = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
        magnitude_db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)
        phase = np.angle(stft)  # Radians, wrapped between -pi and pi

        fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

        magnitude_plot = librosa.display.specshow(
            magnitude_db,
            sr=sample_rate,
            hop_length=hop_length,
            x_axis="time",
            y_axis="hz",
            ax=axes[0],
            cmap="magma",
        )
        axes[0].set_title("Magnitude (dB)")
        fig.colorbar(magnitude_plot, ax=axes[0], format="%+2.0f dB")

        phase_plot = librosa.display.specshow(
            phase,
            sr=sample_rate,
            hop_length=hop_length,
            x_axis="time",
            y_axis="hz",
            ax=axes[1],
            cmap="twilight",
            vmin=-np.pi,
            vmax=np.pi,
        )
        axes[1].set_title("Phase (radians)")
        fig.colorbar(phase_plot, ax=axes[1], ticks=[-np.pi, 0, np.pi])
        axes[1].set_yticks([])

        fig.suptitle(audio_path.name)
        fig.tight_layout()

        graph_path = output_dir / (
            f"{audio_path.stem}_{audio_path.suffix.lstrip('.').lower()}_MG_PH.png"
        )
        fig.savefig(graph_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        saved_graphs.append(graph_path)
        print(f"Saved: {graph_path}")

    return saved_graphs

In [ ]:
graph_paths = plot_magnitude_phase_spectrograms(
    clean_audio_dir,
    output_dir=output_dir / "MG_PH_Spectrogram",
    n_fft=STFT_N_FFT,
    hop_length=STFT_HOP_LENGTH,
)

In [ ]:
def reconstruct_with_zero_phase(
    audio_dir,
    output_dir="/content/outputs/sig1",
    n_fft=STFT_N_FFT,
    hop_length=STFT_HOP_LENGTH,
):
    audio_dir = Path(audio_dir)
    output_dir = Path(output_dir)

    if not audio_dir.is_dir():
        raise NotADirectoryError(f"Audio directory not found: {audio_dir}")

    extensions = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".wma"}
    audio_files = sorted(
        path for path in audio_dir.iterdir()
        if path.is_file() and path.suffix.lower() in extensions
    )

    if not audio_files:
        raise FileNotFoundError(f"No supported audio files found in {audio_dir}")

    output_dir.mkdir(parents=True, exist_ok=True)
    saved_files = []

    for audio_path in audio_files:
        audio, sample_rate = librosa.load(audio_path, sr=None, mono=False)

        # Process mono and multi-channel files while preserving their channel count.
        channels = audio[np.newaxis, :] if audio.ndim == 1 else audio
        reconstructed_channels = []

        for channel in channels:
            stft = librosa.stft(
                channel,
                n_fft=n_fft,
                hop_length=hop_length,
            )
            magnitude = np.abs(stft)

            # A complex STFT with the original magnitude and zero phase.
            zero_phase_stft = magnitude.astype(np.complex64)

            reconstructed = librosa.istft(
                zero_phase_stft,
                n_fft=n_fft,
                hop_length=hop_length,
                length=len(channel),
            )
            reconstructed_channels.append(reconstructed)

        reconstructed_audio = (
            reconstructed_channels[0]
            if audio.ndim == 1
            else np.stack(reconstructed_channels, axis=1)
        )

        output_path = output_dir / f"{audio_path.stem}_zero_phase.wav"
        sf.write(output_path, reconstructed_audio, sample_rate)
        saved_files.append(output_path)
        print(f"Saved reconstructed audio: {output_path}")

    return saved_files

clean_sig1_files = reconstruct_with_zero_phase(
    clean_audio_dir,
    output_dir=output_dir / "sig1",
    n_fft=STFT_N_FFT,
    hop_length=STFT_HOP_LENGTH,
)

In [ ]:
def compare_audio_with_reconstructions(
    audio_dir,
    reconstructed_dir="/content/outputs/sig1",
    plot_dir="/content/outputs/sig1_comparison",
):
    audio_dir = Path(audio_dir)
    reconstructed_dir = Path(reconstructed_dir)
    plot_dir = Path(plot_dir)

    if not audio_dir.is_dir():
        raise NotADirectoryError(f"Original audio directory not found: {audio_dir}")
    if not reconstructed_dir.is_dir():
        raise NotADirectoryError(
            f"Reconstructed audio directory not found: {reconstructed_dir}"
        )

    extensions = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".wma"}
    original_files = sorted(
        path for path in audio_dir.iterdir()
        if path.is_file() and path.suffix.lower() in extensions
    )

    if not original_files:
        raise FileNotFoundError(f"No supported audio files found in {audio_dir}")

    plot_dir.mkdir(parents=True, exist_ok=True)
    results = []

    for original_path in original_files:
        reconstructed_path = (
            reconstructed_dir / f"{original_path.stem}_zero_phase.wav"
        )

        if not reconstructed_path.is_file():
            raise FileNotFoundError(
                f"Reconstruction for {original_path.name} not found: "
                f"{reconstructed_path}"
            )

        original, original_sr = librosa.load(
            original_path, sr=None, mono=False
        )
        reconstructed, reconstructed_sr = librosa.load(
            reconstructed_path, sr=None, mono=False
        )

        if original_sr != reconstructed_sr:
            raise ValueError(
                f"Sample-rate mismatch for {original_path.name}: "
                f"{original_sr} vs {reconstructed_sr}"
            )

        # Normalize array shape to (channels, samples).
        if original.ndim == 1:
            original = original[np.newaxis, :]
        if reconstructed.ndim == 1:
            reconstructed = reconstructed[np.newaxis, :]

        if original.shape[0] != reconstructed.shape[0]:
            raise ValueError(
                f"Channel-count mismatch for {original_path.name}: "
                f"{original.shape[0]} vs {reconstructed.shape[0]}"
            )

        # Compare the shared duration, starting at sample zero.
        sample_count = min(original.shape[1], reconstructed.shape[1])
        original = original[:, :sample_count]
        reconstructed = reconstructed[:, :sample_count]

        error = original - reconstructed
        mse = float(np.mean(error ** 2))
        rmse = float(np.sqrt(mse))

        original_flat = original.ravel()
        reconstructed_flat = reconstructed.ravel()

        if np.std(original_flat) == 0 or np.std(reconstructed_flat) == 0:
            correlation = float("nan")
        else:
            correlation = float(
                np.corrcoef(original_flat, reconstructed_flat)[0, 1]
            )

        duration_seconds = sample_count / original_sr
        results.append({
            "file": original_path.name,
            "correlation": correlation,
            "mse": mse,
            "rmse": rmse,
            "sample_rate": original_sr,
            "compared_duration_seconds": duration_seconds,
        })

        # Limit displayed points for long recordings; metrics use all samples.
        stride = max(1, sample_count // 100_000)
        times = np.arange(0, sample_count, stride) / original_sr

        fig, axes = plt.subplots(
            original.shape[0], 1,
            figsize=(14, max(4, 2.8 * original.shape[0])),
            sharex=True,
            squeeze=False,
        )

        for channel in range(original.shape[0]):
            axes[channel, 0].plot(
                times, original[channel, ::stride],
                label="Original", alpha=0.8, linewidth=0.7,
            )
            axes[channel, 0].plot(
                times, reconstructed[channel, ::stride],
                label="Zero-phase reconstruction", alpha=0.8, linewidth=0.7,
            )
            axes[channel, 0].set_ylabel(f"Channel {channel + 1}")
            axes[channel, 0].grid(True, alpha=0.3)
            axes[channel, 0].legend(loc="upper right")

        axes[-1, 0].set_xlabel("Time (seconds)")
        fig.suptitle(
            f"{original_path.name}\n"
            f"Correlation: {correlation:.4f} | MSE: {mse:.6g} | RMSE: {rmse:.6g}"
        )
        fig.tight_layout()

        plot_path = plot_dir / f"{original_path.stem}_comparison.png"
        fig.savefig(plot_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        print(
            f"{original_path.name}: correlation={correlation:.4f}, "
            f"MSE={mse:.6g}, RMSE={rmse:.6g}\n"
            f"Plot saved: {plot_path}"
        )

    return results

comparison_results = compare_audio_with_reconstructions(
    audio_dir=clean_audio_dir,
    reconstructed_dir=output_dir / "sig1",
)

### Step 2 analysis

STFT breaks the speech into short parts and shows the sound frequencies in each part. Magnitude shows how strong each frequency is. Phase shows the timing of each frequency.

For `sig1`, we kept the original magnitude and changed every phase value to zero. The errors compared with the original audio were:

| Recording | MSE | Correlation |
|---|---:|---:|
| Speech 1 | 0.001669 | -0.0337 |
| Speech 2 | 0.005233 | 0.0258 |

The correlation values were close to zero. This means that magnitude alone did not keep the original waveform. Phase is important for rebuilding the original sound. The clean zero-phase STOI file gave 0.7404 for Speech 1 and 0.7069 for Speech 2. STOI measures speech clarity, while MSE and correlation measure waveform difference.

### Random Phase Reconstruction

In [ ]:
def generate_and_plot_random_phase(
    magnitude,
    filename="audio",
    output_dir="/content/outputs/PH1",
    seed=None,
):
    magnitude = np.asarray(magnitude)

    if magnitude.ndim not in (2, 3):
        raise ValueError(
            "magnitude must have shape (frequency, time) or "
            "(channels, frequency, time)"
        )

    # Use a seed for repeatable results, or leave it as None for a new
    # random phase array each time.
    rng = np.random.default_rng(seed)
    PH1 = rng.uniform(-np.pi, np.pi, size=magnitude.shape)

    channels = PH1[np.newaxis, ...] if PH1.ndim == 2 else PH1
    fig, axes = plt.subplots(
        len(channels), 1,
        figsize=(12, max(4, 3 * len(channels))),
        squeeze=False,
    )

    for channel, ax in enumerate(axes[:, 0]):
        image = ax.imshow(
            channels[channel],
            origin="lower",
            aspect="auto",
            cmap="twilight",
            vmin=-np.pi,
            vmax=np.pi,
        )
        ax.set_title(f"Random phase PH1 — Channel {channel + 1}")
        ax.set_xlabel("Time frame")
        ax.set_ylabel("Frequency bin")
        fig.colorbar(image, ax=ax, label="Phase (radians)")

    fig.tight_layout()

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    plot_path = output_dir / f"{Path(filename).stem}_PH1.png"
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"Plot saved to: {plot_path}")
    return PH1

In [ ]:
def generate_sig2(
    magnitude,
    PH1,
    sample_rate,
    filename="audio",
    output_dir="/content/outputs/sig2",
    n_fft=STFT_N_FFT,
    hop_length=STFT_HOP_LENGTH,
    length=None,
):
    magnitude = np.asarray(magnitude)
    PH1 = np.asarray(PH1)

    if magnitude.shape != PH1.shape:
        raise ValueError(
            f"magnitude and PH1 must have the same shape; "
            f"got {magnitude.shape} and {PH1.shape}"
        )
    if magnitude.ndim not in (2, 3):
        raise ValueError(
            "Expected shape (frequency, time) for mono, or "
            "(channels, frequency, time) for multi-channel audio"
        )

    # Combine magnitude and phase to make the complex-valued STFT.
    complex_stft = magnitude * np.exp(1j * PH1)
    # DC and Nyquist bins must be real for a valid real-valued spectrum.
    if magnitude.ndim == 2:
        complex_stft[0, :] = magnitude[0, :]
        complex_stft[-1, :] = magnitude[-1, :]
    else:
        complex_stft[:, 0, :] = magnitude[:, 0, :]
        complex_stft[:, -1, :] = magnitude[:, -1, :]

    if magnitude.ndim == 2:
        sig2 = librosa.istft(
            complex_stft,
            n_fft=n_fft,
            hop_length=hop_length,
            length=length,
        )
    else:
        sig2 = np.stack([
            librosa.istft(
                channel_stft,
                n_fft=n_fft,
                hop_length=hop_length,
                length=length,
            )
            for channel_stft in complex_stft
        ], axis=1)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / f"{Path(filename).stem}_sig2.wav"
    sf.write(output_path, sig2, sample_rate, subtype="FLOAT")

    print(f"Saved sig2 to: {output_path}")
    return sig2, output_path

In [ ]:
def compare_original_sig1_sig2(
    audio_dir,
    sig1_dir="/content/outputs/sig1",
    sig2_dir="/content/outputs/sig2",
    output_dir="/content/outputs/sig_comparisons",
    plot_seconds=5,
):
    audio_dir = Path(audio_dir)
    sig1_dir = Path(sig1_dir)
    sig2_dir = Path(sig2_dir)
    output_dir = Path(output_dir)

    for folder in (audio_dir, sig1_dir, sig2_dir):
        if not folder.is_dir():
            raise NotADirectoryError(f"Directory not found: {folder}")

    audio_extensions = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".wma"}
    original_files = sorted(
        path for path in audio_dir.iterdir()
        if path.is_file() and path.suffix.lower() in audio_extensions
    )

    if not original_files:
        raise FileNotFoundError(f"No supported audio files found in {audio_dir}")

    output_dir.mkdir(parents=True, exist_ok=True)
    all_results = []

    def calculate_metrics(first, second):
        error = first - second
        mse = float(np.mean(error ** 2))
        rmse = float(np.sqrt(mse))

        if np.std(first) == 0 or np.std(second) == 0:
            correlation = float("nan")
        else:
            correlation = float(np.corrcoef(first, second)[0, 1])

        return {"mse": mse, "rmse": rmse, "correlation": correlation}

    for original_path in original_files:
        sig1_path = sig1_dir / f"{original_path.stem}_zero_phase.wav"
        sig2_path = sig2_dir / f"{original_path.stem}_sig2.wav"

        for generated_path in (sig1_path, sig2_path):
            if not generated_path.is_file():
                raise FileNotFoundError(
                    f"Expected reconstruction for {original_path.name}: "
                    f"{generated_path}"
                )

        original, original_sr = librosa.load(original_path, sr=None, mono=True)
        sig1, sig1_sr = librosa.load(sig1_path, sr=None, mono=True)
        sig2, sig2_sr = librosa.load(sig2_path, sr=None, mono=True)

        if not (original_sr == sig1_sr == sig2_sr):
            raise ValueError(
                f"Sample-rate mismatch for {original_path.name}: "
                f"original={original_sr}, sig1={sig1_sr}, sig2={sig2_sr}"
            )

        # Compare the shared duration, starting at the beginning of each file.
        sample_count = min(len(original), len(sig1), len(sig2))
        if sample_count == 0:
            raise ValueError(f"Audio is empty: {original_path.name}")

        original = original[:sample_count]
        sig1 = sig1[:sample_count]
        sig2 = sig2[:sample_count]

        comparisons = {
            "Original vs sig1": calculate_metrics(original, sig1),
            "Original vs sig2": calculate_metrics(original, sig2),
            "sig1 vs sig2": calculate_metrics(sig1, sig2),
        }

        all_results.append({
            "file": original_path.name,
            "compared_duration_seconds": sample_count / original_sr,
            **comparisons,
        })

        # Plot only the requested portion for readability; metrics use all samples.
        plot_count = min(sample_count, int(plot_seconds * original_sr))
        time = np.arange(plot_count) / original_sr
        pairs = [
            ("Original vs sig1", original[:plot_count], sig1[:plot_count]),
            ("Original vs sig2", original[:plot_count], sig2[:plot_count]),
            ("sig1 vs sig2", sig1[:plot_count], sig2[:plot_count]),
        ]

        fig, axes = plt.subplots(3, 2, figsize=(15, 10))

        for row, (title, first, second) in enumerate(pairs):
            metrics = comparisons[title]

            axes[row, 0].plot(time, first, label=title.split(" vs ")[0], linewidth=0.7)
            axes[row, 0].plot(time, second, label=title.split(" vs ")[1],
                              linewidth=0.7, alpha=0.8)
            axes[row, 0].set_title(title)
            axes[row, 0].set_ylabel("Amplitude")
            axes[row, 0].grid(True, alpha=0.3)
            axes[row, 0].legend(loc="upper right")

            axes[row, 1].plot(time, first - second, color="crimson", linewidth=0.7)
            axes[row, 1].set_title(
                f"Difference | MSE={metrics['mse']:.3g}, "
                f"RMSE={metrics['rmse']:.3g}, "
                f"Correlation={metrics['correlation']:.4f}"
            )
            axes[row, 1].set_ylabel("Amplitude difference")
            axes[row, 1].grid(True, alpha=0.3)

        axes[-1, 0].set_xlabel("Time (seconds)")
        axes[-1, 1].set_xlabel("Time (seconds)")
        fig.suptitle(f"Audio comparison: {original_path.name}")
        fig.tight_layout()

        plot_path = output_dir / f"{original_path.stem}_comparison.png"
        fig.savefig(plot_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        print(f"\n{original_path.name}")
        for name, metrics in comparisons.items():
            print(
                f"  {name}: MSE={metrics['mse']:.6g}, "
                f"RMSE={metrics['rmse']:.6g}, "
                f"correlation={metrics['correlation']:.4f}"
            )
        print(f"  Plot saved: {plot_path}")

    return all_results

### Step 3 analysis

We made a random phase array called `PH1` with the same size as the original phase. We joined it with the original magnitude to create `sig2`. A real audio signal is possible when the positive and negative frequency parts match correctly.

| Recording | MSE | Correlation |
|---|---:|---:|
| Speech 1 | 0.001981 | 0.0118 |
| Speech 2 | 0.006478 | -0.0316 |

The random-phase audio was very different from the original. It had a larger MSE than the zero-phase audio for both recordings.

| Recording | `sig1` MSE | `sig2` MSE |
|---|---:|---:|
| Speech 1 | 0.001669 | 0.001981 |
| Speech 2 | 0.005233 | 0.006478 |

In this experiment, zero phase was closer to the original than random phase. However, neither version matched the original waveform well.

In [ ]:
def create_valid_phase_pattern(n_fft=8, seed=None):
    if n_fft < 4 or n_fft % 2 != 0:
        raise ValueError("n_fft must be an even integer of at least 4.")

    rng = np.random.default_rng(seed)
    phase = np.zeros(n_fft)

    # Generate phases for the positive-frequency bins, then mirror them
    # with opposite signs to satisfy conjugate symmetry.
    for k in range(1, n_fft // 2):
        phase[k] = rng.uniform(-np.pi, np.pi)
        phase[n_fft - k] = -phase[k]

    bins = np.arange(n_fft)

    plt.figure(figsize=(8, 4))
    plt.stem(bins, phase)
    plt.xticks(bins)
    plt.yticks(
        [-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi],
        [r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"],
    )
    plt.xlabel("FFT bin")
    plt.ylabel("Phase (radians)")
    plt.title(f"Valid phase pattern for a {n_fft}-point FFT")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return phase

PH1 = create_valid_phase_pattern(n_fft=8, seed=42)
print("PH1:", PH1)

In [ ]:
def verify_real_valued_reconstruction(phase, seed=42):
    phase = np.asarray(phase, dtype=float)

    if phase.ndim != 1 or len(phase) < 4 or len(phase) % 2 != 0:
        raise ValueError("phase must be a 1D array with an even length of at least 4.")

    n_fft = len(phase)
    rng = np.random.default_rng(seed)

    # Generate magnitudes for DC through Nyquist, then mirror the
    # interior bins so the full magnitude is conjugate-symmetric.
    half_magnitude = rng.uniform(0.5, 2.0, size=n_fft // 2 + 1)
    magnitude = np.empty(n_fft)
    magnitude[:n_fft // 2 + 1] = half_magnitude
    magnitude[n_fft // 2 + 1:] = half_magnitude[1:-1][::-1]

    spectrum = magnitude * np.exp(1j * phase)

    # A real-valued inverse FFT requires conjugate symmetry.
    if not np.allclose(spectrum[1:n_fft // 2],
                       np.conj(spectrum[-1:n_fft // 2:-1])):
        raise ValueError("The phase pattern does not produce conjugate symmetry.")
    if not np.isclose(spectrum[0].imag, 0) or not np.isclose(
        spectrum[n_fft // 2].imag, 0
    ):
        raise ValueError("DC and Nyquist phase must produce real-valued bins.")

    reconstructed = np.fft.ifft(spectrum)
    max_imaginary = float(np.max(np.abs(reconstructed.imag)))
    is_real = np.allclose(reconstructed.imag, 0, atol=1e-10)

    if not is_real:
        raise ValueError(
            f"Reconstruction is not real-valued; max imaginary component "
            f"is {max_imaginary:.3g}."
        )

    reconstructed_real = reconstructed.real
    print(f"Real-valued reconstruction verified: {is_real}")
    print(f"Maximum imaginary component: {max_imaginary:.3g}")

    fig, axes = plt.subplots(2, 1, figsize=(9, 6))

    axes[0].stem(np.arange(n_fft), magnitude)
    axes[0].set_title(f"Conjugate-symmetric magnitude ({n_fft}-point FFT)")
    axes[0].set_xlabel("FFT bin")
    axes[0].set_ylabel("Magnitude")
    axes[0].grid(True, alpha=0.3)

    axes[1].stem(np.arange(n_fft), reconstructed_real)
    axes[1].set_title("Signal reconstructed with inverse FFT")
    axes[1].set_xlabel("Sample")
    axes[1].set_ylabel("Amplitude")
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

    return magnitude, spectrum, reconstructed_real

In [ ]:
def create_random_phase_pattern(n_fft=8, seed=None):
    if n_fft < 1:
        raise ValueError("n_fft must be a positive integer.")

    rng = np.random.default_rng(seed)
    phase = rng.uniform(-np.pi, np.pi, size=n_fft)

    bins = np.arange(n_fft)
    plt.figure(figsize=(8, 4))
    plt.stem(bins, phase)
    plt.xticks(bins)
    plt.yticks(
        [-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi],
        [r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"],
    )
    plt.xlabel("FFT bin")
    plt.ylabel("Phase (radians)")
    plt.title(f"Random phase pattern for a {n_fft}-point FFT")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return phase

PH1 = create_random_phase_pattern(n_fft=8, seed=42)
print("PH1:", PH1)

In [ ]:
def apply_random_phase_to_audio_dir(
    audio_dir,
    output_dir="/content/outputs/sig3",
    n_fft=1024,
    hop_length=256,
    seed=None,
):
    audio_dir = Path(audio_dir)
    output_dir = Path(output_dir)

    if not audio_dir.is_dir():
        raise NotADirectoryError(f"Audio directory not found: {audio_dir}")
    if n_fft % 2:
        raise ValueError("n_fft must be even.")

    extensions = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".wma"}
    audio_files = sorted(
        path for path in audio_dir.iterdir()
        if path.is_file() and path.suffix.lower() in extensions
    )
    if not audio_files:
        raise FileNotFoundError(f"No supported audio files found in {audio_dir}")

    output_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    results = []

    for audio_path in audio_files:
        audio, sample_rate = librosa.load(audio_path, sr=None, mono=False)
        channels = audio[np.newaxis, :] if audio.ndim == 1 else audio
        reconstructed_channels = []

        for channel in channels:
            stft = librosa.stft(
                channel,
                n_fft=n_fft,
                hop_length=hop_length,
            )
            magnitude = np.abs(stft)

            # Randomize the one-sided STFT phase for each bin and frame.
            phase = rng.uniform(-np.pi, np.pi, size=stft.shape)
            phase[0, :] = 0.0                # DC must be real
            phase[-1, :] = 0.0               # Nyquist must be real

            randomized_stft = magnitude * np.exp(1j * phase)

            # Rebuild the full spectrum by mirroring the positive-frequency bins.
            full_spectrum = np.concatenate(
                [
                    randomized_stft,
                    np.conj(randomized_stft[-2:0:-1, :]),
                ],
                axis=0,
            )

            # Check conjugate symmetry and confirm the IFFT is real-valued.
            mirrored_bins_match = np.allclose(
                full_spectrum[1:n_fft // 2, :],
                np.conj(full_spectrum[-1:n_fft // 2:-1, :]),
                atol=1e-10,
            )
            frame_ifft = np.fft.ifft(full_spectrum, axis=0)
            max_imaginary = float(np.max(np.abs(frame_ifft.imag)))

            if not mirrored_bins_match or max_imaginary > 1e-10:
                raise ValueError(
                    f"Conjugate-symmetry verification failed for "
                    f"{audio_path.name}"
                )

            reconstructed = librosa.istft(
                randomized_stft,
                n_fft=n_fft,
                hop_length=hop_length,
                length=len(channel),
            )
            reconstructed_channels.append(reconstructed)

        sig3 = (
            reconstructed_channels[0]
            if audio.ndim == 1
            else np.stack(reconstructed_channels, axis=1)
        )

        output_path = output_dir / f"{audio_path.stem}_sig3.wav"
        sf.write(output_path, sig3, sample_rate)

        results.append({
            "file": audio_path.name,
            "output_path": output_path,
            "one_sided_bins": n_fft // 2 + 1,
            "full_spectrum_bins": n_fft,
            "conjugate_symmetry_verified": True,
            "max_ifft_imaginary": max_imaginary,
        })
        print(f"Saved {output_path}; conjugate symmetry verified.")

    return results

### Step 4 analysis

The phase pattern must obey conjugate symmetry: `PH[N-k] = -PH[k] (mod 2π)`. The magnitude must also be symmetric, and the DC and Nyquist coefficients must be real. Under these conditions, the inverse FFT produces a real-valued signal.

In simple terms, the frequency list must have matching pairs. The pair on the other side uses the opposite phase, which makes the final audio real.

For a 1024-point FFT, 511 inside positive-frequency phase values can be chosen freely in each frame, when the magnitude is not zero. Each value can be any number in a range, so there are infinitely many valid phase patterns.

In [ ]:
def generate_noisy_speech_at_snrs(
    speech_dir,
    noise_dir,
    output_dir="/content/outputs/noisy_speech",
    sample_rate=16000,
    snr_levels_db=(-2, 0, 2, 5, 10),
    seed=42,
):
    speech_dir = Path(speech_dir)
    noise_dir = Path(noise_dir)
    output_dir = Path(output_dir)

    if not speech_dir.is_dir():
        raise NotADirectoryError(f"Speech directory not found: {speech_dir}")
    if not noise_dir.is_dir():
        raise NotADirectoryError(f"Noise directory not found: {noise_dir}")
    if sample_rate <= 0:
        raise ValueError("sample_rate must be positive.")
    if len(snr_levels_db) == 0:
        raise ValueError("snr_levels_db must contain at least one SNR level.")

    extensions = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".wma"}

    speech_files = sorted(
        path for path in speech_dir.iterdir()
        if path.is_file() and path.suffix.lower() in extensions
    )
    noise_files = sorted(
        path for path in noise_dir.iterdir()
        if path.is_file()
        and path.suffix.lower() in extensions
        and path.stem.lower() in NOISE_SOURCE_STEMS
    )

    if not speech_files:
        raise FileNotFoundError(f"No supported speech files found in {speech_dir}")
    if not noise_files:
        raise FileNotFoundError(f"No supported noise files found in {noise_dir}")

    output_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    results = []

    for speech_path in speech_files:
        # librosa resamples to 16 kHz and loads as mono.
        speech, _ = librosa.load(speech_path, sr=sample_rate, mono=True)

        if speech.size == 0:
            raise ValueError(f"Speech file is empty: {speech_path}")

        # Power is calculated over the entire recording, including pauses.
        speech_power = float(np.mean(speech.astype(np.float64) ** 2))
        if speech_power == 0:
            raise ValueError(f"Speech file has zero power: {speech_path}")

        for noise_path in noise_files:
            noise, _ = librosa.load(noise_path, sr=sample_rate, mono=True)

            if noise.size == 0:
                raise ValueError(f"Noise file is empty: {noise_path}")

            # Use a reproducibly selected segment, or repeat shorter noise.
            if noise.size >= speech.size:
                start = int(rng.integers(0, noise.size - speech.size + 1))
                noise_segment = noise[start:start + speech.size]
            else:
                repeats = math.ceil(speech.size / noise.size)
                noise_segment = np.tile(noise, repeats)[:speech.size]

            noise_power = float(np.mean(noise_segment.astype(np.float64) ** 2))
            if noise_power == 0:
                raise ValueError(f"Selected noise segment has zero power: {noise_path}")

            for target_snr_db in snr_levels_db:
                target_snr_db = float(target_snr_db)

                # Scale noise so 10*log10(speech power / noise power) equals target.
                desired_noise_power = speech_power / (10 ** (target_snr_db / 10))
                noise_scale = math.sqrt(desired_noise_power / noise_power)
                scaled_noise = noise_segment * noise_scale
                mixture = speech + scaled_noise

                achieved_snr_db = 10 * math.log10(
                    speech_power
                    / float(np.mean(scaled_noise.astype(np.float64) ** 2))
                )

                snr_tag = (
                    f"{target_snr_db:g}"
                    .replace("-", "m")
                    .replace(".", "p")
                )
                output_path = output_dir / (
                    f"{speech_path.stem}__{noise_path.stem}__snr_{snr_tag}dB.wav"
                )

                # FLOAT avoids clipping or normalization of values outside [-1, 1].
                sf.write(output_path, mixture, sample_rate, subtype="FLOAT")

                results.append({
                    "speech_file": speech_path.name,
                    "noise_type": noise_path.stem,
                    "target_snr_db": target_snr_db,
                    "achieved_snr_db": achieved_snr_db,
                    "snr_error_db": achieved_snr_db - target_snr_db,
                    "sample_rate_hz": sample_rate,
                    "samples": int(speech.size),
                    "duration_seconds": speech.size / sample_rate,
                    "output_file": str(output_path),
                })

    csv_path = output_dir / "snr_verification.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)

    print(f"Generated {len(results)} noisy audio files in: {output_dir}")
    print(f"SNR verification table saved to: {csv_path}")
    return results

### Step 5 analysis

We made phase-scrambled speech by keeping the magnitude and changing the phase to random values. It may sound unclear, but random phase does not always make every word impossible to understand. A listener still needs to test it.

We added white, machinegun, and babble noise to both recordings at -2, 0, 2, 5, and 10 dB SNR. This made 30 noisy recordings. The noisy reconstruction CSV files have 30 rows each.

STOI values are interpreted on a 0–1 scale: higher values indicate better predicted intelligibility. Lower SNR means stronger noise relative to speech, so intelligibility is generally expected to improve as SNR increases.

In simple terms, a STOI score closer to 1 means clearer speech, while a score closer to 0 means less clear speech.

The checked reconstruction STOI CSVs contained 30 rows each. Their mean scores were:

| Speech | Noisy zero-phase mean STOI | Noisy random-phase mean STOI |
|---|---:|---:|
| Speech 1 | 0.5866 | 0.6204 |
| Speech 2 | 0.5485 | 0.5907 |

These are STOI scores, not direct waveform matches. MSE and correlation are better measures for checking the phase reconstruction.

In [ ]:
def design_and_apply_bandpass(
    audio,
    sample_rate=16000,
    lowcut_hz=300.0,
    highcut_hz=3400.0,
    order=4,
):
    """Apply a zero-phase Butterworth band-pass filter to mono audio."""
    audio = np.asarray(audio, dtype=np.float64)

    if audio.ndim != 1:
        raise ValueError("audio must be a one-dimensional mono signal.")
    if audio.size == 0:
        raise ValueError("audio cannot be empty.")
    if sample_rate <= 0:
        raise ValueError("sample_rate must be positive.")
    if order < 1:
        raise ValueError("order must be at least 1.")
    if not 0 < lowcut_hz < highcut_hz < sample_rate / 2:
        raise ValueError(
            "Cutoffs must satisfy 0 < lowcut_hz < highcut_hz < Nyquist "
            f"({sample_rate / 2:g} Hz)."
        )

    sos = butter(
        order,
        [lowcut_hz, highcut_hz],
        btype="bandpass",
        fs=sample_rate,
        output="sos",
    )
    filtered_audio = sosfiltfilt(sos, audio)
    return filtered_audio, sos

In [ ]:
def plot_bandpass_frequency_response(
    sos,
    sample_rate=16000,
    lowcut_hz=300.0,
    highcut_hz=3400.0,
    worN=4096,
    output_path=None,
):
    """Plot the single-pass and forward-backward filter responses."""
    if sample_rate <= 0:
        raise ValueError("sample_rate must be positive.")
    if not 0 < lowcut_hz < highcut_hz < sample_rate / 2:
        raise ValueError(
            "Cutoffs must satisfy 0 < lowcut_hz < highcut_hz < Nyquist "
            f"({sample_rate / 2:g} Hz)."
        )
    if worN < 2:
        raise ValueError("worN must be at least 2.")

    frequencies, response = sosfreqz(sos, worN=worN, fs=sample_rate)
    magnitude = np.maximum(np.abs(response), np.finfo(float).tiny)

    one_pass_db = 20 * np.log10(magnitude)
    forward_backward_db = 40 * np.log10(magnitude)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(frequencies, one_pass_db, label="Single-pass response")
    ax.plot(
        frequencies,
        forward_backward_db,
        "--",
        label="Effective forward-backward response",
    )
    ax.axvline(lowcut_hz, color="gray", linestyle=":", label="Cutoff frequencies")
    ax.axvline(highcut_hz, color="gray", linestyle=":")
    ax.set_xlim(0, sample_rate / 2)
    ax.set_ylim(-100, 5)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Magnitude (dB)")
    ax.set_title("Butterworth band-pass frequency response")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=150, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    return frequencies, one_pass_db, forward_backward_db

In [ ]:
def filter_noisy_recordings(
    input_dir="/content/outputs/noisy_speech",
    output_dir="/content/outputs/filtered_speech",
    lowcut_hz=300.0,
    highcut_hz=3400.0,
    order=4,
):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    if not input_dir.is_dir():
        raise NotADirectoryError(f"Noisy recordings directory not found: {input_dir}")
    if order < 1:
        raise ValueError("order must be at least 1.")

    wav_files = sorted(
        path for path in input_dir.iterdir()
        if path.is_file() and path.suffix.lower() == ".wav"
    )
    if not wav_files:
        raise FileNotFoundError(f"No WAV files found in {input_dir}")

    output_dir.mkdir(parents=True, exist_ok=True)
    results = []

    for input_path in wav_files:
        audio, sample_rate = sf.read(
            input_path,
            dtype="float64",
            always_2d=True,
        )

        nyquist_hz = sample_rate / 2
        if not 0 < lowcut_hz < highcut_hz < nyquist_hz:
            raise ValueError(
                f"For {input_path.name}, require 0 < lowcut < highcut < "
                f"Nyquist ({nyquist_hz:g} Hz); got {lowcut_hz:g}, {highcut_hz:g}."
            )

        sos = butter(
            order,
            [lowcut_hz, highcut_hz],
            btype="bandpass",
            fs=sample_rate,
            output="sos",
        )

        filtered = np.empty_like(audio)
        for channel in range(audio.shape[1]):
            filtered[:, channel] = sosfiltfilt(sos, audio[:, channel])

        output_path = output_dir / f"{input_path.stem}_filtered.wav"
        sf.write(output_path, filtered, sample_rate, subtype="FLOAT")

        results.append({
            "input_file": input_path.name,
            "output_file": output_path.name,
            "sample_rate_hz": sample_rate,
            "channels": audio.shape[1],
            "samples": audio.shape[0],
            "lowcut_hz": lowcut_hz,
            "highcut_hz": highcut_hz,
            "order": order,
        })
        print(f"Saved filtered recording: {output_path}")

    manifest_path = output_dir / "filter_manifest.csv"
    with manifest_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)

    print(f"Filtered {len(results)} recordings; originals were not modified.")
    print(f"Manifest saved to: {manifest_path}")
    return results

In [ ]:
def listen_to_filter_comparison(
    noisy_dir="/content/outputs/noisy_speech",
    filtered_dir="/content/outputs/filtered_speech",
    filename=None,
):
    noisy_dir = Path(noisy_dir)
    filtered_dir = Path(filtered_dir)

    if not noisy_dir.is_dir():
        raise NotADirectoryError(f"Noisy audio directory not found: {noisy_dir}")
    if not filtered_dir.is_dir():
        raise NotADirectoryError(f"Filtered audio directory not found: {filtered_dir}")

    noisy_files = sorted(
        path for path in noisy_dir.iterdir()
        if path.is_file() and path.suffix.lower() == ".wav"
    )
    if not noisy_files:
        raise FileNotFoundError(f"No WAV files found in {noisy_dir}")

    if filename is None:
        print("Available noisy recordings:")
        for path in noisy_files:
            print(f" - {path.name}")
        print("Call again with filename='the_file.wav' to play a pair.")
        return

    noisy_path = noisy_dir / Path(filename).name
    if not noisy_path.is_file():
        raise FileNotFoundError(f"Noisy recording not found: {noisy_path}")

    filtered_path = filtered_dir / f"{noisy_path.stem}_filtered.wav"
    if not filtered_path.is_file():
        raise FileNotFoundError(f"Filtered version not found: {filtered_path}")

    noisy_audio, noisy_sr = sf.read(noisy_path, dtype="float32", always_2d=True)
    filtered_audio, filtered_sr = sf.read(
        filtered_path, dtype="float32", always_2d=True
    )

    if noisy_sr != filtered_sr:
        raise ValueError(
            f"Sample-rate mismatch: noisy={noisy_sr} Hz, filtered={filtered_sr} Hz"
        )
    if noisy_audio.shape != filtered_audio.shape:
        raise ValueError(
            f"Audio shape mismatch: noisy={noisy_audio.shape}, "
            f"filtered={filtered_audio.shape}"
        )

    display(Markdown(f"### Original noisy recording: `{noisy_path.name}`"))
    display(Audio(noisy_audio, rate=noisy_sr, normalize=False))

    display(Markdown("### Band-pass filtered recording"))
    display(Audio(filtered_audio, rate=filtered_sr, normalize=False))

    return noisy_audio, filtered_audio, noisy_sr

In [ ]:
def evaluate_stoi_before_after_filtering(
    speech_dir,
    noisy_dir="outputs/noisy_speech",
    filtered_dir="outputs/filtered_speech",
    output_dir="/content/outputs/stoi_filter_comparison",
    sample_rate=16000,
):
    speech_dir = Path(speech_dir)
    noisy_dir = Path(noisy_dir)
    filtered_dir = Path(filtered_dir)
    output_dir = Path(output_dir)

    for folder in (speech_dir, noisy_dir, filtered_dir):
        if not folder.is_dir():
            raise NotADirectoryError(f"Directory not found: {folder}")

    audio_extensions = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".wma"}
    clean_files = sorted(
        path for path in speech_dir.iterdir()
        if path.is_file() and path.suffix.lower() in audio_extensions
    )
    if not clean_files:
        raise FileNotFoundError(f"No clean speech files found in {speech_dir}")

    clean_by_stem = {path.stem: path for path in clean_files}
    if len(clean_by_stem) != len(clean_files):
        raise ValueError("Clean speech files must have unique filenames without extensions.")

    noisy_files = sorted(
        path for path in noisy_dir.iterdir()
        if path.is_file() and path.suffix.lower() == ".wav"
    )
    if not noisy_files:
        raise FileNotFoundError(f"No noisy WAV files found in {noisy_dir}")

    # Expected filename: speech_stem__noise_stem__snr_m2dB.wav
    # The "m" in m2 means -2.
    name_pattern = re.compile(
        r"^(?P<prefix>.+)__snr_(?P<snr>m?\d+(?:p\d+)?)dB$"
    )
    results = []

    for noisy_path in noisy_files:
        match = name_pattern.match(noisy_path.stem)
        if not match:
            raise ValueError(f"Unexpected noisy filename format: {noisy_path.name}")

        speech_stem, separator, noise_type = match.group("prefix").partition("__")
        if not separator or not speech_stem or not noise_type:
            raise ValueError(f"Cannot parse speech/noise names: {noisy_path.name}")
        if speech_stem not in clean_by_stem:
            raise FileNotFoundError(
                f"No clean reference for '{speech_stem}' ({noisy_path.name})"
            )

        snr_text = match.group("snr").replace("m", "-").replace("p", ".")
        target_snr_db = float(snr_text)

        filtered_path = filtered_dir / f"{noisy_path.stem}_filtered.wav"
        if not filtered_path.is_file():
            raise FileNotFoundError(f"Filtered recording not found: {filtered_path}")

        clean, clean_sr = sf.read(clean_by_stem[speech_stem], dtype="float64")
        noisy, noisy_sr = sf.read(noisy_path, dtype="float64")
        filtered, filtered_sr = sf.read(filtered_path, dtype="float64")

        # Convert stereo inputs to mono consistently for STOI.
        if clean.ndim == 2:
            clean = np.mean(clean, axis=1)
        if noisy.ndim == 2:
            noisy = np.mean(noisy, axis=1)
        if filtered.ndim == 2:
            filtered = np.mean(filtered, axis=1)

        if not (clean_sr == noisy_sr == filtered_sr == sample_rate):
            raise ValueError(
                f"Expected all files at {sample_rate} Hz for {noisy_path.name}; "
                f"got clean={clean_sr}, noisy={noisy_sr}, filtered={filtered_sr}."
            )
        if not (len(clean) == len(noisy) == len(filtered)):
            raise ValueError(
                f"Length mismatch for {noisy_path.name}: clean={len(clean)}, "
                f"noisy={len(noisy)}, filtered={len(filtered)}."
            )

        score_before = float(stoi(clean, noisy, sample_rate, extended=False))
        score_after = float(stoi(clean, filtered, sample_rate, extended=False))

        results.append({
            "speech": speech_stem,
            "noise_type": noise_type,
            "target_snr_db": target_snr_db,
            "stoi_before_filtering": score_before,
            "stoi_after_filtering": score_after,
            "stoi_change": score_after - score_before,
            "noisy_file": noisy_path.name,
            "filtered_file": filtered_path.name,
        })

    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / "stoi_before_after.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)

    speech_names = sorted({row["speech"] for row in results})
    noise_types = sorted({row["noise_type"] for row in results})

    # Keep each speech recording visible instead of hiding differences in an average.
    for speech_name in speech_names:
        fig, (score_ax, change_ax) = plt.subplots(
            2, 1, figsize=(10, 9), sharex=True
        )

        for noise_type in noise_types:
            subset = sorted(
                (
                    row for row in results
                    if row["speech"] == speech_name
                    and row["noise_type"] == noise_type
                ),
                key=lambda row: row["target_snr_db"],
            )
            if not subset:
                continue

            snrs = [row["target_snr_db"] for row in subset]
            before = [row["stoi_before_filtering"] for row in subset]
            after = [row["stoi_after_filtering"] for row in subset]
            changes = [row["stoi_change"] for row in subset]

            score_ax.plot(snrs, before, marker="o", label=f"{noise_type}: noisy")
            score_ax.plot(
                snrs, after, marker="s", linestyle="--",
                label=f"{noise_type}: filtered",
            )
            change_ax.plot(snrs, changes, marker="o", label=noise_type)

        score_ax.set_title(f"STOI before and after filtering — {speech_name}")
        score_ax.set_ylabel("STOI (higher generally indicates better intelligibility)")
        score_ax.set_ylim(0, 1)
        score_ax.grid(True, alpha=0.3)
        score_ax.legend(ncol=2, fontsize="small")

        change_ax.axhline(0, color="black", linewidth=0.8)
        change_ax.set_title("Filtering impact (filtered STOI − noisy STOI)")
        change_ax.set_xlabel("Target SNR (dB)")
        change_ax.set_ylabel("STOI change")
        change_ax.grid(True, alpha=0.3)
        change_ax.legend(title="Noise type")

        fig.tight_layout()
        plot_path = output_dir / f"{speech_name}_stoi_comparison.png"
        fig.savefig(plot_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

    # Compare average filter impact across the noise types and all conditions.
    mean_changes = {
        noise_type: float(np.mean([
            row["stoi_change"] for row in results
            if row["noise_type"] == noise_type
        ]))
        for noise_type in noise_types
    }

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(mean_changes.keys(), mean_changes.values())
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title("Mean STOI change after filtering by noise type")
    ax.set_ylabel("Mean STOI change (filtered − noisy)")
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()

    impact_plot = output_dir / "mean_filtering_impact_by_noise.png"
    fig.savefig(impact_plot, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Evaluated {len(results)} noisy/filtered recordings.")
    print(f"Scores saved to: {csv_path}")
    print(f"Plots saved in: {output_dir}")
    print("Mean STOI change by noise type:")
    for noise_type, change in mean_changes.items():
        print(f"  {noise_type}: {change:+.4f}")

    return results, mean_changes

In [ ]:
def reconstruct_filtered_recording(
    input_path,
    output_path,
    lowcut_hz=300.0,
    highcut_hz=3400.0,
    order=4,
):
    """Band-pass filter one WAV recording and save the filtered copy."""
    input_path = Path(input_path)
    output_path = Path(output_path)

    if not input_path.is_file():
        raise FileNotFoundError(f"Input recording not found: {input_path}")
    if order < 1:
        raise ValueError("order must be at least 1.")

    audio, sample_rate = sf.read(
        input_path,
        dtype="float64",
        always_2d=True,
    )

    nyquist_hz = sample_rate / 2
    if not 0 < lowcut_hz < highcut_hz < nyquist_hz:
        raise ValueError(
            f"Cutoffs must be between 0 and Nyquist ({nyquist_hz:g} Hz); "
            f"got {lowcut_hz:g} and {highcut_hz:g}."
        )

    sos = butter(
        order,
        [lowcut_hz, highcut_hz],
        btype="bandpass",
        fs=sample_rate,
        output="sos",
    )

    reconstructed = np.empty_like(audio)
    for channel in range(audio.shape[1]):
        reconstructed[:, channel] = sosfiltfilt(sos, audio[:, channel])

    output_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(output_path, reconstructed, sample_rate, subtype="FLOAT")
    return output_path


def reconstruct_all_noise_conditions(
    noisy_dir="outputs/noisy_speech",
    output_dir="/content/outputs/reconstructed_speech",
    lowcut_hz=300.0,
    highcut_hz=3400.0,
    order=4,
):
    """Create a filtered reconstruction for every noisy WAV in the folder."""
    noisy_dir = Path(noisy_dir)
    output_dir = Path(output_dir)

    if not noisy_dir.is_dir():
        raise NotADirectoryError(f"Noisy recordings directory not found: {noisy_dir}")

    wav_files = sorted(
        path for path in noisy_dir.iterdir()
        if path.is_file() and path.suffix.lower() == ".wav"
    )
    if not wav_files:
        raise FileNotFoundError(f"No WAV files found in {noisy_dir}")

    output_dir.mkdir(parents=True, exist_ok=True)
    records = []

    for input_path in wav_files:
        output_path = output_dir / f"{input_path.stem}_filtered.wav"

        reconstruct_filtered_recording(
            input_path=input_path,
            output_path=output_path,
            lowcut_hz=lowcut_hz,
            highcut_hz=highcut_hz,
            order=order,
        )

        records.append({
            "input_file": input_path.name,
            "reconstructed_file": output_path.name,
            "lowcut_hz": lowcut_hz,
            "highcut_hz": highcut_hz,
            "filter_order": order,
        })
        print(f"Saved: {output_path}")

    manifest_path = output_dir / "reconstruction_manifest.csv"
    with manifest_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=records[0].keys())
        writer.writeheader()
        writer.writerows(records)

    print(f"Processed {len(records)} noise/SNR conditions.")
    print(f"Original noisy files were not modified.")
    print(f"Manifest saved to: {manifest_path}")

    return records

In [ ]:
def plot_magnitude_phase_for_noisy_signals(
    noisy_dir="outputs/noisy_speech",
    output_dir="/content/outputs/noisy_magnitude_phase",
    nperseg=1024,
    noverlap=768,
):
    noisy_dir = Path(noisy_dir)
    output_dir = Path(output_dir)

    if not noisy_dir.is_dir():
        raise NotADirectoryError(f"Noisy recordings directory not found: {noisy_dir}")
    if nperseg < 2:
        raise ValueError("nperseg must be at least 2.")
    if not 0 <= noverlap < nperseg:
        raise ValueError("noverlap must satisfy 0 <= noverlap < nperseg.")

    audio_files = sorted(
        path for path in noisy_dir.iterdir()
        if path.is_file() and path.suffix.lower() == ".wav"
    )
    if not audio_files:
        raise FileNotFoundError(f"No WAV files found in {noisy_dir}")

    output_dir.mkdir(parents=True, exist_ok=True)
    manifest = []

    for audio_path in audio_files:
        audio, sample_rate = sf.read(
            audio_path,
            dtype="float64",
            always_2d=True,
        )
        mono_audio = np.mean(audio, axis=1)

        segment_length = min(nperseg, len(mono_audio))
        if segment_length < 2:
            raise ValueError(f"Audio is too short to analyze: {audio_path}")
        overlap = min(noverlap, segment_length - 1)

        frequencies, times, spectrum = stft(
            mono_audio,
            fs=sample_rate,
            window="hann",
            nperseg=segment_length,
            noverlap=overlap,
            boundary="zeros",
            padded=True,
        )

        magnitude_db = 20 * np.log10(np.maximum(np.abs(spectrum), 1e-10))
        phase = np.angle(spectrum)

        fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

        magnitude_plot = axes[0].pcolormesh(
            times, frequencies, magnitude_db,
            shading="auto", cmap="magma",
        )
        axes[0].set_title("Magnitude spectrogram (dB)")
        axes[0].set_xlabel("Time (s)")
        axes[0].set_ylabel("Frequency (Hz)")
        fig.colorbar(magnitude_plot, ax=axes[0], label="Magnitude (dB)")

        phase_plot = axes[1].pcolormesh(
            times, frequencies, phase,
            shading="auto", cmap="twilight",
            vmin=-np.pi, vmax=np.pi,
        )
        axes[1].set_title("Phase spectrogram (radians)")
        axes[1].set_xlabel("Time (s)")
        axes[1].set_ylabel("Frequency (Hz)")
        fig.colorbar(phase_plot, ax=axes[1], label="Phase (radians)")

        fig.suptitle(f"{audio_path.name} — magnitude and phase")
        fig.tight_layout()

        plot_path = output_dir / f"{audio_path.stem}_magnitude_phase.png"
        fig.savefig(plot_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        manifest.append({
            "audio_file": audio_path.name,
            "sample_rate_hz": sample_rate,
            "channels": audio.shape[1],
            "duration_seconds": len(mono_audio) / sample_rate,
            "plot_file": plot_path.name,
        })
        print(f"Saved: {plot_path}")

    manifest_path = output_dir / "plot_manifest.csv"
    with manifest_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=manifest[0].keys())
        writer.writeheader()
        writer.writerows(manifest)

    print(f"Created {len(manifest)} plots in {output_dir}")
    return manifest

### Step 6 analysis

We repeated the STFT and phase steps for the noisy recordings. We also used a fourth-order Butterworth band-pass filter from 300 Hz to 3,400 Hz. The filter was run forward and backward so it did not add a time shift.

| Noise type | Mean STOI change after filtering |
|---|---:|
| Babble | -0.0472 |
| Machinegun | -0.0568 |
| White | -0.0833 |

STOI became lower for all three noise types. White noise had the biggest drop, and babble had the smallest drop. The filter removed some useful speech sounds, while noise inside the kept frequency range remained. So, this filter did not improve the average speech score.

## Complete directory-based Colab run

This section runs the experiment for every normalized clean file and every noise file found in the configured folders.

In [ ]:
def wav_files(folder):
    folder = Path(folder)
    return sorted(path for path in folder.iterdir() if path.is_file() and path.suffix.lower() == ".wav")

clean_files = wav_files(clean_audio_dir)
noise_files = [path for path in wav_files(noise_dir) if path.stem.lower() in NOISE_SOURCE_STEMS]
if not clean_files:
    raise FileNotFoundError(f"No normalized clean WAV files found in {clean_audio_dir}")
if not noise_files:
    raise FileNotFoundError(f"No WAV noise files found in {noise_dir}")

snr_levels = (-2, 0, 2, 5, 10)
noisy_dir = output_dir / "noisy_speech"
filtered_dir = output_dir / "filtered_speech"

noisy_records = generate_noisy_speech_at_snrs(
    speech_dir=clean_audio_dir,
    noise_dir=noise_dir,
    output_dir=noisy_dir,
    sample_rate=TARGET_SAMPLE_RATE,
    snr_levels_db=snr_levels,
    seed=42,
)
expected_noisy_count = len(clean_files) * len(noise_files) * len(snr_levels)
assert len(noisy_records) == expected_noisy_count, (
    f"Expected {expected_noisy_count} noisy files, got {len(noisy_records)}"
)

filter_records = filter_noisy_recordings(
    input_dir=noisy_dir,
    output_dir=filtered_dir,
    lowcut_hz=300.0,
    highcut_hz=3400.0,
    order=4,
)
assert len(filter_records) == expected_noisy_count

sos = butter(
    4,
    [300.0, 3400.0],
    btype="bandpass",
    fs=TARGET_SAMPLE_RATE,
    output="sos",
)
plot_bandpass_frequency_response(
    sos,
    sample_rate=TARGET_SAMPLE_RATE,
    lowcut_hz=300.0,
    highcut_hz=3400.0,
    output_path=output_dir / "bandpass_frequency_response.png",
)

# Use one consistent STFT configuration for clean and noisy reconstruction.
def reconstruct_directory_with_phase(input_dir, output_subdir, mode, seed=42):
    if mode not in {"zero", "random"}:
        raise ValueError("mode must be 'zero' or 'random'")
    input_dir = Path(input_dir)
    output_subdir = Path(output_subdir)
    output_subdir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    records = []

    for input_path in wav_files(input_dir):
        audio, sample_rate = librosa.load(input_path, sr=None, mono=True)
        spectrum = librosa.stft(
            audio, n_fft=STFT_N_FFT, hop_length=STFT_HOP_LENGTH, window="hann"
        )
        magnitude = np.abs(spectrum)
        if mode == "zero":
            phase = np.zeros_like(magnitude)
        else:
            phase = rng.uniform(-np.pi, np.pi, size=magnitude.shape)
            phase[0, :] = 0.0
            phase[-1, :] = 0.0
        complex_spectrum = magnitude * np.exp(1j * phase)
        if mode == "random":
            full_spectrum = np.concatenate(
                [complex_spectrum, np.conj(complex_spectrum[-2:0:-1, :])], axis=0
            )
            if not np.allclose(full_spectrum[1:STFT_N_FFT // 2, :], np.conj(full_spectrum[-1:STFT_N_FFT // 2:-1, :]), atol=1e-10):
                raise ValueError(f"Conjugate symmetry failed for {input_path.name}")
            if np.max(np.abs(np.fft.ifft(full_spectrum, axis=0).imag)) > 1e-10:
                raise ValueError(f"Non-real reconstruction spectrum for {input_path.name}")
        reconstructed = librosa.istft(
            complex_spectrum,
            n_fft=STFT_N_FFT,
            hop_length=STFT_HOP_LENGTH,
            window="hann",
            length=len(audio),
        )
        output_path = output_subdir / f"{input_path.stem}_{mode}_phase.wav"
        sf.write(output_path, reconstructed, sample_rate, subtype="FLOAT")
        records.append({
            "input_file": input_path.name,
            "output_file": output_path.name,
            "mode": mode,
            "sample_rate_hz": sample_rate,
            "samples": len(audio),
        })

    if not records:
        raise FileNotFoundError(f"No WAV files found in {input_dir}")
    manifest_path = output_subdir / "reconstruction_manifest.csv"
    with manifest_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=records[0].keys())
        writer.writeheader()
        writer.writerows(records)
    return records

clean_zero_dir = output_dir / "sig1"
clean_random_dir = output_dir / "sig2"
noisy_zero_dir = output_dir / "noisy_zero_phase"
noisy_random_dir = output_dir / "noisy_random_phase"
clean_zero_records = reconstruct_directory_with_phase(clean_audio_dir, clean_zero_dir, "zero")
clean_random_records = reconstruct_directory_with_phase(clean_audio_dir, clean_random_dir, "random")
noisy_zero_records = reconstruct_directory_with_phase(noisy_dir, noisy_zero_dir, "zero")
noisy_random_records = reconstruct_directory_with_phase(noisy_dir, noisy_random_dir, "random")
assert len(clean_zero_records) == len(clean_files)
assert len(clean_random_records) == len(clean_files)
assert len(noisy_zero_records) == expected_noisy_count
assert len(noisy_random_records) == expected_noisy_count

plot_magnitude_phase_for_noisy_signals(
    noisy_dir=noisy_dir,
    output_dir=output_dir / "noisy_magnitude_phase",
    nperseg=STFT_N_FFT,
    noverlap=STFT_N_FFT - STFT_HOP_LENGTH,
)

def write_reconstruction_metrics(clean_dir, candidate_dirs, output_path):
    clean_by_stem = {path.stem: path for path in wav_files(clean_dir)}
    rows = []
    for label, candidate_dir in candidate_dirs.items():
        for candidate_path in wav_files(candidate_dir):
            if not candidate_path.stem.endswith(("_zero_phase", "_random_phase")):
                continue
            speech_stem = candidate_path.stem.rsplit("_", 2)[0]
            reference_path = clean_by_stem[speech_stem]
            reference, reference_sr = sf.read(reference_path, dtype="float64")
            candidate, candidate_sr = sf.read(candidate_path, dtype="float64")
            length = min(len(reference), len(candidate))
            reference = reference[:length]
            candidate = candidate[:length]
            error = reference - candidate
            mse = float(np.mean(error ** 2))
            correlation = float("nan") if np.std(reference) == 0 or np.std(candidate) == 0 else float(np.corrcoef(reference, candidate)[0, 1])
            rows.append({
                "file": reference_path.name,
                "reconstruction": label,
                "mse": mse,
                "rmse": float(np.sqrt(mse)),
                "correlation": correlation,
                "sample_rate_hz": reference_sr,
                "samples_compared": length,
            })
    output_path = Path(output_path)
    with output_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    return rows

clean_reconstruction_metrics = write_reconstruction_metrics(
    clean_audio_dir,
    {"zero_phase": clean_zero_dir, "random_phase": clean_random_dir},
    output_dir / "clean_reconstruction_metrics.csv",
)

def evaluate_candidate_stoi(candidate_dir, clean_dir, output_path):
    clean_by_stem = {path.stem: path for path in wav_files(clean_dir)}
    rows = []
    for candidate_path in wav_files(candidate_dir):
        if not candidate_path.stem.endswith(("_zero_phase", "_random_phase")):
            continue
        base = candidate_path.stem.rsplit("_", 2)[0]
        speech_stem = base.split("__", 1)[0]
        if speech_stem not in clean_by_stem:
            raise FileNotFoundError(f"No clean reference for {candidate_path.name}")
        clean, clean_sr = sf.read(clean_by_stem[speech_stem], dtype="float64")
        candidate, candidate_sr = sf.read(candidate_path, dtype="float64")
        if clean.ndim == 2:
            clean = np.mean(clean, axis=1)
        if candidate.ndim == 2:
            candidate = np.mean(candidate, axis=1)
        if clean_sr != candidate_sr or clean_sr != TARGET_SAMPLE_RATE:
            raise ValueError(f"Sample-rate mismatch for {candidate_path.name}")
        length = min(len(clean), len(candidate))
        rows.append({
            "candidate_file": candidate_path.name,
            "speech": speech_stem,
            "stoi": float(stoi(clean[:length], candidate[:length], clean_sr, extended=False)),
            "samples_compared": length,
        })
    if not rows:
        raise FileNotFoundError(f"No candidate WAV files found in {candidate_dir}")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    return rows

clean_zero_stoi = evaluate_candidate_stoi(clean_zero_dir, clean_audio_dir, output_dir / "stoi_clean_zero_phase.csv")
clean_random_stoi = evaluate_candidate_stoi(clean_random_dir, clean_audio_dir, output_dir / "stoi_clean_random_phase.csv")
noisy_zero_stoi = evaluate_candidate_stoi(noisy_zero_dir, clean_audio_dir, output_dir / "stoi_noisy_zero_phase.csv")
noisy_random_stoi = evaluate_candidate_stoi(noisy_random_dir, clean_audio_dir, output_dir / "stoi_noisy_random_phase.csv")

filter_stoi_results, mean_filter_changes = evaluate_stoi_before_after_filtering(
    speech_dir=clean_audio_dir,
    noisy_dir=noisy_dir,
    filtered_dir=filtered_dir,
    output_dir=output_dir / "stoi_filter_comparison",
    sample_rate=TARGET_SAMPLE_RATE,
)
assert len(filter_stoi_results) == expected_noisy_count

print("Complete experiment finished.")
print(f"Clean files: {len(clean_files)}")
print(f"Noise files: {len(noise_files)}")
print(f"Noisy/filtered conditions: {expected_noisy_count}")
print(f"Outputs saved under: {output_dir}")

## Overall conclusion

The clean-audio tests show that phase is important when rebuilding speech. Both zero-phase and random-phase signals had correlations close to zero. Random phase had the larger MSE for both recordings.

In simple terms, changing the phase changed the shape and timing of the speech wave.

The noise experiment used 30 conditions across three noise types and five SNR levels. The reported filter comparison produced negative mean STOI changes for babble, machinegun, and white noise. This indicates that the selected 300–3,400 Hz band-pass filter did not improve average predicted intelligibility, because it removed useful speech components while leaving in-band noise.
